In [7]:
import os

for root, dir, files in os.walk('/kaggle'):
    print ( root, dir, files)

/kaggle ['lib', 'input', 'working'] []
/kaggle/lib ['kaggle'] []
/kaggle/lib/kaggle [] ['gcp.py']
/kaggle/input ['datasets'] []
/kaggle/input/datasets ['deoussama'] []
/kaggle/input/datasets/deoussama ['3d-smart-v2', 'lois-des-obligation-et-contrats'] []
/kaggle/input/datasets/deoussama/3d-smart-v2 ['loc_pages'] ['loc_articles.json', 'loc_embeddings.npy', 'loc_ocr_pages.json']
/kaggle/input/datasets/deoussama/3d-smart-v2/loc_pages [] ['page-010.jpg', 'page-167.jpg', 'page-045.jpg', 'page-216.jpg', 'page-077.jpg', 'page-190.jpg', 'page-179.jpg', 'page-081.jpg', 'page-023.jpg', 'page-164.jpg', 'page-003.jpg', 'page-220.jpg', 'page-160.jpg', 'page-040.jpg', 'page-121.jpg', 'page-011.jpg', 'page-102.jpg', 'page-257.jpg', 'page-152.jpg', 'page-266.jpg', 'page-001.jpg', 'page-209.jpg', 'page-058.jpg', 'page-183.jpg', 'page-094.jpg', 'page-088.jpg', 'page-143.jpg', 'page-109.jpg', 'page-147.jpg', 'page-259.jpg', 'page-067.jpg', 'page-261.jpg', 'page-036.jpg', 'page-202.jpg', 'page-051.jpg', '

In [8]:
BASE_DIR = '/kaggle/input/datasets/deoussama/3d-smart-v2'

os.makedirs(BASE_DIR, exist_ok=True)

OCR_CACHE      = os.path.join(BASE_DIR, "loc_ocr_pages.json")
ARTICLES_CACHE = os.path.join(BASE_DIR, "loc_articles.json")
EMBED_CACHE    = os.path.join(BASE_DIR, "loc_embeddings.npy")
IMG_DIR        = os.path.join(BASE_DIR, "loc_pages")

os.makedirs(IMG_DIR, exist_ok=True)

In [10]:
# =====================================================================
# CELL 3 — Nettoyage OCR + Parsing des articles
#
# Ce que cette cellule fait:
#   1. Définit toutes les fonctions de nettoyage (clean_page, etc.)
#   2. Définit LegalArticle et LOCParser
#   3. Charge le JSON OCR, nettoie, parse -> variable `articles`
#
# Variable produite pour les cellules suivantes:
#   articles : list[LegalArticle]  (1258 articles du LOC)
# =====================================================================

import re, os
import json
from dataclasses import dataclass, asdict
from typing import Optional

# ─────────────────────────────────────────────────────────────────
# PARTIE A — Filtres de nettoyage par ligne
# ─────────────────────────────────────────────────────────────────

# Déclencheur HARD de note de bas de page.
# ATTENTION : "1 - الأهلية للالتزام" = CONTENU d'article (liste numérotée) -> NE PAS filtrer
# On filtre UNIQUEMENT les lignes contenant des verbes de référence légale
# qui n'apparaissent JAMAIS dans le corps d'un article.
_HARD_FOOTNOTE = re.compile(
    r"""^\d{1,3}\s*[-\u2013]\s*
    (?:
        ورد\s+في\s+النص       |
        وردت\s+في\s+النص      |
        قارن\s+مع\s+          |
        انظر\s+(?:الفصل|المادة|ظهير)    |
        تمم\s+(?:القسم|الفصل|الباب)     |
        تممت\s+               |
        أضاف\s+               |
        أضيف\s+               |
        تم\s+تغيير            |
        تم\s+تتميم            |
        تم\s+إضافة            |
        المادة\s+\d+\s+من\s+(?:مدونة|القانون|قانون)  |
        مقارنة\s+مع\s+النص    |
        سقطت\s+من\s+الترجمة   |
        تتحدث\s+بعض\s+فصول   |
        المقصود\s+بالقانون
    )""",
    re.VERBOSE | re.UNICODE,
)

# Lignes de continuation de notes (références Bulletin Officiel, dahirs)
_FOOTNOTE_CONTINUATION = re.compile(
    r"""(?:
        الجريدة\s+الرسمية              |
        ظهير\s+شريف\s+رقم             |
        بتنفيذه\s+ظهير                |
        صادر\s+في\s+\d+\s+من          |
        بتاريخ\s+\d+\s+من\s+(?:ذي|ربيع|شعبان|رمضان|شوال|ذو|جمادى|محرم|صفر)  |
        \bص\s*\d{3,}\b                |
        عدد\s+\d{4,}\s+بتاريخ         |
        رقم\s+[\d.]+\s+(?:يتعلق|بمثابة|المتعلق)
    )""",
    re.VERBOSE | re.UNICODE,
)

_ARTICLE_HEADER = re.compile(r"^الفصل\s*[\d]")


def _is_page_header(s):
    """En-tête ministériel haut de page (39 variantes OCR couvertes)."""
    return bool(
        re.search(r"(?:المملكة|الدسايكة|الساكة|المساكة|الم\s+اكد)", s)
        and re.search(r"(?:وزارة|العذز|العذل|العذ\b|مكيرية|مديرية)", s)
    )

def _is_page_number(s):
    return bool(re.match(r"^[-\u2013\s]*\d{1,3}[-\u2013\s]*$", s))

def _is_stray_number(s):
    return bool(re.match(r"^\d{1,3}$", s))

def _is_garbled_latin(s):
    """Ratio caractères latins > 45% = citation française OCR-ée."""
    if len(s) < 8:
        return False
    return len(re.findall(r"[a-zA-Z]", s)) / len(s) > 0.45

def _is_editorial_rewrite(s):
    """Reformulation éditoriale (correction de traduction, pas du texte de loi)."""
    return bool(re.search(r"وبذلك\s+يمكن\s+(?:صياغة|تحديد)", s))


def clean_page(text):
    """
    Nettoie une page OCR du LOC.

    Supprime : en-tête ministériel, numéros de page, notes de bas de page,
    lignes de références JO, garbled Latin, reformulations éditoriales.

    Conserve : corps des articles, listes numérotées DANS les articles
    (ex: "1 - الأهلية للالتزام"), titres structurels.
    """
    lines = text.split("\n")
    clean = []
    in_footnote_zone = False

    for line in lines:
        s = line.strip()
        if not s:
            continue

        # Filtres absolus
        if _is_page_header(s):      continue
        if _is_page_number(s):      continue
        if _is_stray_number(s):     continue
        if _is_garbled_latin(s):    continue
        if _is_editorial_rewrite(s):
            in_footnote_zone = True
            continue

        # Nouveau article → reset zone notes
        if _ARTICLE_HEADER.match(s):
            in_footnote_zone = False

        # Déclencheur HARD de note (verbes de référence légale uniquement)
        if _HARD_FOOTNOTE.match(s):
            in_footnote_zone = True

        if in_footnote_zone:
            continue

        # Référence légale hors-zone (sécurité supplémentaire)
        if re.match(r"^\d{1,3}\s*[-\u2013]", s) and _FOOTNOTE_CONTINUATION.search(s):
            continue

        clean.append(s)

    return "\n".join(clean)


# ─────────────────────────────────────────────────────────────────
# PARTIE B — Normalisation du texte
# ─────────────────────────────────────────────────────────────────

_OCR_QUOTES  = re.compile(r'[""\u201c\u201d]')
_SUPERSCRIPT = re.compile(r'([\u0600-\u06FF])[\'\d]{1,3}(?=\s|$|[،؛.])')
_MULTI_SPACE = re.compile(r'[ \t]{2,}')


def normalize_line(s):
    """Normalisation légère : guillemets OCR, superscripts parasites, espaces."""
    s = _OCR_QUOTES.sub("", s)
    s = _SUPERSCRIPT.sub(r"\1", s)
    s = _MULTI_SPACE.sub(" ", s)
    return s.strip()


# ─────────────────────────────────────────────────────────────────
# PARTIE C — Correction des numéros d'articles contaminés
#
# OCR colle parfois le chiffre superscript au numéro d'article :
#   "الفصل2901197" = "الفصل 290" + référence "1197"
#   "الفصل 2.1197" = "الفصل 2.1" + référence "197"
#   "الفصل4277"   = "الفصل 427" + référence "7"
#
# Le LOC va de 1 à ~1250 (articles purs)
# Composites réels : X-Y (ex: 3-106, 618-20, 417-1) et X.Y (ex: 2.1)
# ─────────────────────────────────────────────────────────────────

_LOC_MAX = 1260


def clean_article_num(raw):
    """Récupère le vrai numéro d'article depuis un token OCR potentiellement contaminé."""
    s = raw.strip()

    # Cas X.Y (point — seulement "2.1" dans le LOC)
    m = re.match(r"^(\d+)\.(\d+)$", s)
    if m:
        x_str, y_str = m.group(1), m.group(2)
        x, y = int(x_str), int(y_str)
        if x <= _LOC_MAX:
            if y <= 9:
                return s                           # "2.1" → valide
            else:
                return f"{x_str}.{y_str[0]}"      # "2.1197" → "2.1"

    # Cas X-Y (tiret — composites réels du LOC)
    m = re.match(r"^(\d+)-(\d+)$", s)
    if m:
        x_str, y_str = m.group(1), m.group(2)
        x, y = int(x_str), int(y_str)
        if x <= _LOC_MAX and y <= _LOC_MAX:
            return s                               # "3-106", "617-20" → valide
        if x <= _LOC_MAX:
            for l in range(len(y_str) - 1, 0, -1):
                if int(y_str[:l]) <= _LOC_MAX:
                    return f"{x_str}-{y_str[:l]}"
        for l in range(len(x_str) - 1, 0, -1):
            if 0 < int(x_str[:l]) <= _LOC_MAX:
                return f"{x_str[:l]}-{y_str[:3] if len(y_str) > 3 else y_str}"

    # Cas entier pur
    m = re.match(r"^(\d+)$", s)
    if m:
        n_str = m.group(1)
        if int(n_str) <= _LOC_MAX:
            return s
        for l in range(len(n_str) - 1, 0, -1):
            if 0 < int(n_str[:l]) <= _LOC_MAX:
                return n_str[:l]

    return s  # fallback


# ─────────────────────────────────────────────────────────────────
# PARTIE D — Dataclass LegalArticle
# ─────────────────────────────────────────────────────────────────

@dataclass
class LegalArticle:
    """
    Un article du LOC après nettoyage.

    Champs :
        article_num     → numéro textuel ("1", "2.1", "417-1", "618-20")
        article_num_int → premier entier pour tri/filtre
        text            → corps de l'article nettoyé
        book            → الكتاب courant
        section         → القسم / الباب courant
        subsection      → الفرع courant
        char_count      → longueur (pour déduplication)
    """
    article_num:     str
    article_num_int: int
    text:            str
    book:            str = ""
    section:         str = ""
    subsection:      str = ""
    char_count:      int = 0

    def to_chunk_text(self):
        """
        Texte enrichi pour l'embedding.
        Format : [الفصل N] [الكتاب X] [القسم Y]\n<corps>
        """
        header = f"[الفصل {self.article_num}]"
        if self.book:       header += f" [{self.book}]"
        if self.section:    header += f" [{self.section}]"
        if self.subsection: header += f" [{self.subsection}]"
        return f"{header}\n{self.text}"


# ─────────────────────────────────────────────────────────────────
# PARTIE E — LOCParser
# ─────────────────────────────────────────────────────────────────

class LOCParser:
    """
    Parseur structuré du قانون الالتزامات والعقود.
    Pipeline : nettoyer → splitter sur الفصل N → extraire contexte → dédupliquer → trier.
    """

    _ART_SPLIT = re.compile(
        r"(?:^|\n)(الفصل\s*[\d]+(?:[.\-][\d]+)*)",
        re.MULTILINE,
    )
    _ART_NUM   = re.compile(r"الفصل\s*([\d]+(?:[.\-][\d]+)*)")
    _BOOK_RE   = re.compile(r"الكتاب\s+\S+(?:\s+\S+)?")
    _SECT_RE   = re.compile(r"(?:القسم|الباب)\s+\S+(?:\s+\S+)?")
    _SUB_RE    = re.compile(r"الفرع\s+\S+(?:\s+\S+)?")
    _HDR_NOISE = re.compile(
        r"(?:المملكة|الدسايكة).*?(?:التشريع|التشيوع|العشيوم)\n?",
        re.DOTALL,
    )

    @staticmethod
    def _to_int(s):
        m = re.search(r"\d+", s)
        return int(m.group()) if m else 0

    def parse(self, ocr_pages):
        """Parse depuis {page_num: raw_text}. Retourne list[LegalArticle]."""
        pages_sorted = sorted(ocr_pages.keys(), key=lambda k: int(k))
        full = "\n".join(clean_page(ocr_pages[k]) for k in pages_sorted)

        parts = self._ART_SPLIT.split(full)
        arts = []
        cur_book = cur_sect = cur_sub = ""

        if parts:
            bm = self._BOOK_RE.search(parts[0])
            if bm: cur_book = bm.group().strip()
            sm = self._SECT_RE.search(parts[0])
            if sm: cur_sect = sm.group().strip()

        i = 1
        while i < len(parts) - 1:
            header = parts[i].strip()
            body   = parts[i + 1] if i + 1 < len(parts) else ""

            bm = self._BOOK_RE.search(body)
            if bm: cur_book = bm.group().strip()
            sm = self._SECT_RE.search(body)
            if sm: cur_sect = sm.group().strip()
            sub = self._SUB_RE.search(body)
            if sub: cur_sub = sub.group().strip()

            nm = self._ART_NUM.search(header)
            if nm:
                num_str    = clean_article_num(nm.group(1))
                clean_body = self._HDR_NOISE.sub("", body).strip()
                clean_body = re.sub(r"\n{3,}", "\n\n", clean_body)
                clean_body = "\n".join(
                    normalize_line(ln)
                    for ln in clean_body.split("\n")
                    if ln.strip()
                )
                if len(clean_body) > 20:
                    arts.append(LegalArticle(
                        article_num=num_str,
                        article_num_int=self._to_int(num_str),
                        text=clean_body,
                        book=cur_book,
                        section=cur_sect,
                        subsection=cur_sub,
                        char_count=len(clean_body),
                    ))
            i += 2

        seen = {}
        for a in arts:
            if a.article_num not in seen or a.char_count > seen[a.article_num].char_count:
                seen[a.article_num] = a

        return sorted(seen.values(), key=lambda a: (a.article_num_int, a.article_num))

    def save(self, articles, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump([asdict(a) for a in articles], f, ensure_ascii=False, indent=2)
        print(f"Sauvegarde : {len(articles)} articles → {path}")

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            return [LegalArticle(**d) for d in json.load(f)]


# ─────────────────────────────────────────────────────────────────
# PARTIE F — Exécution : produire la variable `articles`
#
# C'est ici que `articles` est créé et rendu disponible
# pour toutes les cellules suivantes (4, 5, 6, 7...).
# ─────────────────────────────────────────────────────────────────

parser = LOCParser()

if os.path.exists(ARTICLES_CACHE):
    # Rechargement depuis cache (évite de re-parser à chaque session)
    articles = parser.load(ARTICLES_CACHE)
    print(f"Cache chargé : {len(articles)} articles depuis {ARTICLES_CACHE}")
else:
    # Premier lancement : parser depuis le JSON OCR
    print("Parsing depuis le JSON OCR...")
    with open(OCR_CACHE, "r", encoding="utf-8") as f:
        ocr_pages = json.load(f)
    articles = parser.parse(ocr_pages)
    parser.save(articles, ARTICLES_CACHE)

# ─────────────────────────────────────────────────────────────────
# Statistiques de vérification
# ─────────────────────────────────────────────────────────────────
total_chars = sum(a.char_count for a in articles)
print(f"\nRésultats :")
print(f"  Articles extraits  : {len(articles)}")
print(f"  Plage              : الفصل {articles[0].article_num} → الفصل {articles[-1].article_num}")
print(f"  Longueur moyenne   : {total_chars // len(articles)} chars/article")
print(f"  Livres détectés    : {len(set(a.book for a in articles if a.book))}")
short = sum(1 for a in articles if a.char_count < 50)
if short:
    print(f"  Articles courts    : {short} (vérifier manuellement)")

# Aperçu de 3 articles pour vérification visuelle
print("\n--- الفصل 2 ---")
a2 = next((a for a in articles if a.article_num == "2"), None)
if a2: print(a2.to_chunk_text())

print("\n--- الفصل 77 ---")
a77 = next((a for a in articles if a.article_num == "77"), None)
if a77: print(a77.to_chunk_text()[:300])

print("\n✅ Variable `articles` prête pour les cellules 4, 5, 6, 7")


Cache chargé : 1258 articles depuis /kaggle/input/datasets/deoussama/3d-smart-v2/loc_articles.json

Résultats :
  Articles extraits  : 1258
  Plage              : الفصل 1 → الفصل 1250
  Longueur moyenne   : 262 chars/article
  Livres détectés    : 5
  Articles courts    : 15 (vérifier manuellement)

--- الفصل 2 ---
[الفصل 2] [الكتاب الأول منه] [الباب الأول: الالتزامات]
الأركان اللازمة لصحة الالتزامات الناشئة عن التعبير عن الإرادةهى:
1 - الأهليةللالتزام؛
2 - تعبير صحيح عن الإرادة يقع على العناصر الأساسيةللالتزام؛
4 - سبب مشروعللالتزام.

--- الفصل 77 ---
[الفصل 77] [الكتاب الأول من] [الباب الثالث: الالتزامات] [الفرع الرابع: أحكاممتفرقة]
كل فعل ارتكبه الإنسان عن بينة واختيارء ومن غير أن يسمح له به القانون»فأحدث
السبب المباشر في حصولالضرر.
وكل شرط مخالف لذلك يكون عديمالأثر.

✅ Variable `articles` prête pour les cellules 4, 5, 6, 7


In [20]:
import subprocess

def run(cmd, label=""):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"{'OK' if r.returncode==0 else 'ERR'} | {label or cmd[:65]}")
    if r.returncode != 0: print(f"   {r.stderr[-200:]}")

# run("apt-get install -y -q tesseract-ocr tesseract-ocr-ara poppler-utils", "tesseract-ocr Arabic + poppler-utils")
# run("pip install -q pytesseract Pillow", "pytesseract + Pillow")
run("pip install -q transformers sentence-transformers", "transformers")
run("pip install -q torch --index-url https://download.pytorch.org/whl/cu118", "torch CUDA")
run("pip install -q faiss-gpu-cu11", "faiss-gpu")
run("pip install -q rank_bm25", "rank_bm25")
run("pip install llama-cpp-python \
      --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121",
    "llama-cpp-python CUDA")
run("pip install -q huggingface_hub", "huggingface_hub")

OK | transformers
OK | torch CUDA
OK | faiss-gpu
OK | rank_bm25
OK | llama-cpp-python CUDA
OK | huggingface_hub


In [22]:
# CELL 4 - Embeddings Arabic BERT + Index FAISS GPU
#
# Requiert : articles (Cell 3), EMBED_CACHE, os
# Produit  : embed_model, embeddings, index, DIM, np
#
# Modele : CAMeL-Lab/camel-bert-base-sts
#   Fine-tune Arabic Semantic Textual Similarity
#   768 dims, ~110M params, rapide sur T4
#   Meilleur que mBERT/LaBSE pour arabe mono-langue

import torch
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

print("\nChargement CAMeL-Lab/camel-bert-base-sts...")
# embed_model = SentenceTransformer("CAMeL-Lab/camel-bert-base-sts", device=DEVICE)
embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", device=DEVICE)


chunk_texts = [a.to_chunk_text() for a in articles]

if os.path.exists(EMBED_CACHE):
    embeddings = np.load(EMBED_CACHE)
    print(f"Embeddings depuis cache : {embeddings.shape}")
else:
    print(f"Encodage de {len(chunk_texts)} articles...")
    embeddings = embed_model.encode(
        chunk_texts, batch_size=64, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True
    )
    np.save(EMBED_CACHE, embeddings)
    print(f"Embeddings sauvegardes : {embeddings.shape}")

DIM       = embeddings.shape[1]
cpu_index = faiss.IndexFlatIP(DIM)
cpu_index.add(embeddings.astype(np.float32))

if DEVICE == "cuda":
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    print(f"\nFAISS GPU : {index.ntotal} vecteurs x {DIM} dims")
else:
    index = cpu_index
    print(f"\nFAISS CPU : {index.ntotal} vecteurs x {DIM} dims")

print("embed_model et index prets pour Cell 5")

Device : cuda
  GPU  : Tesla T4
  VRAM : 15.6 GB

Chargement CAMeL-Lab/camel-bert-base-sts...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings depuis cache : (1258, 768)

FAISS GPU : 1258 vecteurs x 768 dims
embed_model et index prets pour Cell 5


In [23]:
# CELL 5 - BM25 sparse + Hybrid Retriever RRF - << RETRIEVAL >>
#
# Requiert : articles, index, embed_model, np  (Cells 3 & 4)
# Produit  : retriever
#
# RRF formula : score(d) = SUM [ 1 / (k + rank_i(d)) ]  k=60
# Dense BERT = semantique | BM25 = termes exacts juridiques

import re
from dataclasses import dataclass
from rank_bm25 import BM25Okapi

_STOP = {
    "من","في","على","عن","الى","هذا","هذه","التي","الذي",
    "وان","ان","لا","ما","مع","او","ولا","لم","كان","يكون",
    "اذا","هو","هي","وهو","وهي","ذلك","تلك"
}

def tok(text):
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[^\u0600-\u06FF\s\d]", " ", text)
    return [t for t in text.split() if len(t) > 2 and t not in _STOP]

print("Construction de l'index BM25...")
tokenized = [tok(a.to_chunk_text()) for a in articles]
bm25      = BM25Okapi(tokenized)
print(f"BM25 : {len(tokenized)} docs | {sum(len(t) for t in tokenized)//len(tokenized)} tokens/doc")

@dataclass
class RetrievedArticle:
    article:     object
    dense_score: float
    bm25_score:  float
    rrf_score:   float
    dense_rank:  int
    bm25_rank:   int

class HybridRetriever:

    def __init__(self, articles, faiss_index, embed_model, bm25_index, k=60):
        self.articles = articles
        self.index    = faiss_index
        self.embed    = embed_model
        self.bm25     = bm25_index
        self.k        = k

    def _dense(self, query, n=20):
        v = self.embed.encode(
            [query], normalize_embeddings=True, convert_to_numpy=True
        ).astype(np.float32)
        scores, idxs = self.index.search(v, n)
        return list(zip(idxs[0].tolist(), scores[0].tolist()))

    def _sparse(self, query, n=20):
        tokens = tok(query)
        if not tokens: return []
        scores = self.bm25.get_scores(tokens)
        top_i  = np.argsort(scores)[::-1][:n]
        return [(int(i), float(scores[i])) for i in top_i]

    def _fuse(self, dense_r, bm25_r, top_k=10):
        rrf, d_rk, b_rk, d_sc, b_sc = {}, {}, {}, {}, {}
        for rk, (i, s) in enumerate(dense_r):
            rrf[i]  = rrf.get(i, 0) + 1.0 / (self.k + rk + 1)
            d_rk[i] = rk + 1;  d_sc[i] = s
        for rk, (i, s) in enumerate(bm25_r):
            rrf[i]  = rrf.get(i, 0) + 1.0 / (self.k + rk + 1)
            b_rk[i] = rk + 1;  b_sc[i] = s
        top = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            RetrievedArticle(
                article=self.articles[i],
                dense_score=d_sc.get(i, 0.0), bm25_score=b_sc.get(i, 0.0),
                rrf_score=rrf[i], dense_rank=d_rk.get(i, 999), bm25_rank=b_rk.get(i, 999)
            )
            for i in top if 0 <= i < len(self.articles)
        ]

    def retrieve(self, query, top_k=5):
        return self._fuse(self._dense(query, 20), self._sparse(query, 20), top_k)

retriever = HybridRetriever(
    articles=articles, faiss_index=index,
    embed_model=embed_model, bm25_index=bm25
)
print("\nHybrid Retriever pret (Dense + BM25 + RRF k=60)")

print("\nTest : 'شروط صحة العقد'")
for r in retriever.retrieve("شروط صحة العقد", top_k=3):
    print(f"  الفصل {r.article.article_num:>6} | RRF={r.rrf_score:.4f} | {r.article.text[:60]}...")

Construction de l'index BM25...
BM25 : 1258 docs | 44 tokens/doc

Hybrid Retriever pret (Dense + BM25 + RRF k=60)

Test : 'شروط صحة العقد'
  الفصل     20 | RRF=0.0320 | لا يكون العقد تاما إذا احتفظ المتعاقدان صراحة بشروط معينة لك...
  الفصل    393 | RRF=0.0164 | تنقضي الالتزامات التعاقدية» إذا ارتضى المتعاقدان عقب إبرام ا...
  الفصل   1-65 | RRF=0.0164 | مع مراعاة أحكام هذا الباب» تخضع صحة العقد المبرم بشكل إلكترو...


In [24]:
# CELL 6 - Chargement Qwen2.5-7B-Instruct Q4_K_M via llama-cpp
#
# Produit : llm
#
# Comparatif LLMs arabes open-source (MMLU-AR) :
#   Qwen2.5-7B-Instruct  ~72%  <- CHOIX OPTIMAL
#   LLaMA-3.1-8B-Instruct ~61%
#   Mistral-7B-Instruct   ~58%
#
# Pourquoi llama-cpp et non Ollama dans Colab ?
#   API Python directe, n_gpu_layers=-1, pas de daemon reseau.
#   Ollama = meilleur sur machine locale persistante.

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

MODEL_REPO = "pbhappliedsystems/qwen-2.5-7B-instruct-gguf-Q4-K-M"
MODEL_FILE = "qwen-2.5-7B-instruct-gguf-Q4-K-M.gguf"
MODEL_PATH = f"/content/{MODEL_FILE}"

if not os.path.exists(MODEL_PATH):
    print(f"Telechargement {MODEL_FILE} (~4.4 GB)...")
    hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, local_dir="/content")
    print("Telechargement termine")
else:
    print(f"Modele deja present : {MODEL_PATH}")

print("\nChargement en VRAM GPU (n_gpu_layers=-1)...")
llm = Llama(
    model_path   = MODEL_PATH,
    n_gpu_layers = -1,
    n_ctx        = 4096,
    n_batch      = 512,
    verbose      = False,
    seed         = 42,
    chat_format  = "chatml"
)
print("Qwen2.5-7B-Instruct charge en GPU")

resp = llm.create_chat_completion(
    messages=[{"role": "user", "content": " مستشار قانوني مغربي"}],
    max_tokens=10
)
print(f"Test LLM : {resp['choices'][0]['message']['content']}")
print("\nllm pret pour Cell 7")

Telechargement qwen-2.5-7B-instruct-gguf-Q4-K-M.gguf (~4.4 GB)...


qwen-2.5-7B-instruct-gguf-Q4-K-M.gguf:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

Telechargement termine

Chargement en VRAM GPU (n_gpu_layers=-1)...


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Qwen2.5-7B-Instruct charge en GPU
Test LLM : نعم، يمكنني مساعدتك كمستشار

llm pret pour Cell 7


In [25]:
_EXPAND_TMPL = """
أنت خبير قانوني في القانون المغربي.

السؤال:
"{q}"

المطلوب:
- توليد 3 استفسارات بحث قانونية مختلفة
- يجب أن تكون:
  * دقيقة قانونياً
  * متنوعة (تعريف / شرط / تطبيق)
  * تحتوي مصطلحات قانونية

[OUTPUT]
JSON فقط:
[
  "استفسار قانوني 1",
  "استفسار قانوني 2",
  "استفسار قانوني 3"
]
"""

_ANSWER_TMPL = """
[ROLE]
أنت خبير قانوني متخصص في قانون الالتزامات والعقود المغربي (ظهير 9 رمضان 1331).

[CONTEXT]
المقتطفات القانونية التالية هي المصدر الوحيد للحقيقة:
\"\"\"
{ctx}
\"\"\"

[QUESTION]
{q}

[STRICT RULES]
- استخدم فقط المعلومات الموجودة في [CONTEXT]
- ممنوع استخدام أي معرفة خارجية
- إذا لم تجد الجواب في النص → قل صراحة: "المعطيات غير كافية"
- لا تخمن، لا تستنتج بدون دليل صريح
- كل معلومة يجب أن تكون مرتبطة بنص قانوني واضح

[CITATION RULE]
- بعد كل فكرة قانونية، اكتب: [الفصل X]
- لا تذكر أي فصل غير موجود في السياق
- يمكن ذكر عدة فصول إذا لزم

[REASONING METHOD]
1. حدد الفصول المرتبطة بالسؤال
2. استخرج القاعدة القانونية منها
3. اربطها بالسؤال بشكل مباشر
4. اشرح بدون إضافة خارج النص

[OUTPUT FORMAT]
- جواب منظم وواضح
- فقرات قصيرة
- لغة عربية قانونية دقيقة

[ANSWER]
"""

In [26]:
# CELL 7 - Agentic RAG Engine
#
# Requiert : llm (Cell 6), retriever (Cell 5)
# Produit  : rag
#
# Boucle agentique :
#   1. Query expansion  -> LLM genere 3 sous-requetes arabes
#   2. Multi-retrieval  -> retrieval hybride pour chaque sous-requete
#   3. Deduplication    -> fusion + re-ranking par score RRF
#   4. Generation       -> Qwen repond en citant [الفصل X]
#   5. Post-traitement  -> extraction citations + score confiance

import re
import json
from dataclasses import dataclass


@dataclass
class RAGResponse:
    query:              str
    expanded_queries:   list
    retrieved_articles: list
    answer:             str
    cited_articles:     list
    confidence:         str

class AgenticLegalRAG:

    def __init__(self, llm, retriever):
        self.llm = llm
        self.ret = retriever

    def _call_llm(self, prompt, max_tokens=512, temperature=0.1):
        resp = self.llm.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens, temperature=temperature, top_p=0.9
        )
        return resp["choices"][0]["message"]["content"].strip()

    def _expand(self, query):
        raw = self._call_llm(_EXPAND_TMPL.format(q=query), 200, 0.4)
        m   = re.search(r"\[.*?\]", raw, re.DOTALL)
        if m:
            try:
                exp = json.loads(m.group())
                return [x for x in exp if isinstance(x, str)][:3]
            except Exception:
                pass
        return [query]

    def _multi_retrieve(self, queries, top_k=5):
        seen = {}
        for q in queries:
            for r in self.ret.retrieve(q, top_k=top_k):
                k = r.article.article_num
                if k not in seen or r.rrf_score > seen[k].rrf_score:
                    seen[k] = r
        return sorted(seen.values(), key=lambda x: x.rrf_score, reverse=True)[:top_k + 3]

    def query(self, user_q, verbose=True):
        SEP = "=" * 65
        if verbose:
            print(f"\n{SEP}")
            print(f"Question : {user_q}")

        expanded  = self._expand(user_q)
        if verbose:
            print("\n[1] Sous-requetes :")
            for i, q in enumerate(expanded, 1): print(f"    {i}. {q}")

        retrieved = self._multi_retrieve([user_q] + expanded)
        if verbose:
            print("\n[2] Articles retrouves :")
            for r in retrieved[:5]:
                print(f"    الفصل {r.article.article_num:>6} | RRF={r.rrf_score:.4f} | {r.article.text[:55]}...")

        ctx_parts = []
        for r in retrieved[:5]:
            hdr = f"\u0627\u0644\u0641\u0635\u0644 {r.article.article_num}"
            if r.article.book: hdr += f" | {r.article.book}"
            ctx_parts.append(f"{'='*50}\n{hdr}\n{r.article.text}")
        ctx = "\n".join(ctx_parts)

        if verbose: print("\n[3] Generation en cours...")
        answer = self._call_llm(_ANSWER_TMPL.format(q=user_q, ctx=ctx), 900, 0.1)

        cited      = list(set(re.findall(r"\u0627\u0644\u0641\u0635\u0644\s+([\d]+(?:[\-.]\d+)?)", answer)))
        top_rrf    = retrieved[0].rrf_score if retrieved else 0
        confidence = "HIGH" if top_rrf > 0.025 else "MEDIUM" if top_rrf > 0.015 else "LOW"

        if verbose:
            print(f"\n{SEP}\nREPONSE [{confidence}]\n{SEP}\n{answer}")
            print(f"\nArticles cites : {cited}")

        return RAGResponse(
            query=user_q, expanded_queries=expanded,
            retrieved_articles=[{"num": r.article.article_num, "rrf": round(r.rrf_score, 4),
                                  "preview": r.article.text[:100]} for r in retrieved[:5]],
            answer=answer, cited_articles=cited, confidence=confidence
        )

rag = AgenticLegalRAG(llm=llm, retriever=retriever)
print("Agentic RAG pret !")

Agentic RAG pret !


In [29]:
USE_CASES = [
    "ما هي الشروط القانونية لصحة عقد البيع؟",
    # "متى يكون العقد باطلا أو قابلا للإبطال؟",
    # "ما هي أحكام المسؤولية المدنية وشروط التعويض؟",
    # "ما هي مدد التقادم في قانون الالتزامات؟",
]

responses = []
for i, case in enumerate(USE_CASES, 1):
    print(f"\n{'#'*70}\nUSE CASE {i}/{len(USE_CASES)}")
    responses.append(rag.query(case, verbose=True))

print(f"\n{len(responses)} use cases traites.")
print("Reponses stockees dans `responses`")


######################################################################
USE CASE 1/1

Question : ما هي الشروط القانونية لصحة عقد البيع؟

[1] Sous-requetes :
    1. ما هي التعريف القانوني للعقد المبرم بين المبيع والمشتري في القانون المغربي؟
    2. ما هي الشروط القانونية اللازمة لصحة عقد البيع وفقاً للقانون المغربي؟
    3. كيف يمكن تطبيق الشروط القانونية لصحة عقد البيع في حالة خلاف بين طرفي العقد؟

[2] Articles retrouves :
    الفصل    516 | RRF=0.0296 | الالتزام بتسليم الشيء يشمل أيضا توابعه؛ وفقا لما يقضي ب...
    الفصل    600 | RRF=0.0287 | إذا سمي الاتفاق (ببيع الثُنْيَا) مع كونه يتضمن في الحقي...
    الفصل    601 | RRF=0.0285 | يسوغ أن يشترط في عقد البيع ثبوت الحق للمشتري أو للبائع ...
    الفصل    534 | RRF=0.0275 | ويلتزم البائع أيضا بقوة القانون بأن يضمن للمشتري الاستح...
    الفصل    504 | RRF=0.0164 | يجب أن يحصل التسليم فور إبرام العقد, إلا ما تقتضيه طبيع...

[3] Generation en cours...

REPONSE [HIGH]
عقد البيع يجب أن يلتزم به الشروط القانونية التالية:

1. **التسليم فور إبرام 

In [ ]:
# CELL 9 - Interface interactive
# Requiert : rag (Cell 7)
# Saisir 'خروج' ou Ctrl+C pour quitter

print("Systeme RAG — قانون الالتزامات والعقود")
print("Saisir votre question en arabe | 'خروج' pour quitter\n")

while True:
    try:
        q = input("Question : ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nFin de session.")
        break
    if not q: continue
    if q in ["\u062e\u0631\u0648\u062c", "exit", "quit"]:
        print("Au revoir.")
        break
    try:
        rag.query(q, verbose=True)
    except Exception as e:
        print(f"[ERREUR] {e}")